# 3.4 Dynamic Models — Switching LLMs Per Request

## What you will learn in this notebook

- Why you might want to use **different models for different requests**
- How **`@wrap_model_call`** middleware intercepts and modifies LLM calls
- How to switch models dynamically based on **conversation length or state**
- How to verify **which model actually answered** via `response_metadata`

---

### Why switch models dynamically?

Not every conversation turn needs the same model. Using a powerful (expensive) model for simple questions wastes money. Using a weak (cheap) model for complex reasoning produces poor results.

**Dynamic model selection** lets you route automatically:

| Condition | Use this model | Why |
|---|---|---|
| Short conversation (< 10 messages) | `gpt-5-nano` (fast, cheap) | Simple context, routine task |
| Long conversation (10+ messages) | `claude-sonnet-4-5` (powerful) | Long context needs better reasoning |
| User is on free tier | Cheap model | Cost control |
| User is on pro tier | Premium model | Better experience |
| Code generation task | Coding-specialist model | Domain expertise |

### What is `@wrap_model_call`?

`@wrap_model_call` decorates a function that wraps **every LLM call** inside the agent loop. It receives:
- `request` — the full context of the LLM call (messages, tools, model, state, runtime)
- `handler` — a callable to actually execute the LLM call

You can inspect `request`, modify it via `request.override(...)`, and then call `handler(request)` to proceed.

In [2]:
from dataclasses import dataclass

# ============================================================
# CELL 1: Load Environment Variables
# ============================================================
# Requires OPENAI_API_KEY (gpt-5-nano) and ANTHROPIC_API_KEY
# (claude-sonnet-4-5). Both providers are used in this notebook.

from dotenv import load_dotenv

load_dotenv()

True

In [3]:
# ============================================================
# CELL 2: Define the Dynamic Model Selection Middleware
# ============================================================
# We pre-initialise both models once at module load time.
# This avoids re-creating the model object on every call (expensive).
#
# @wrap_model_call wraps every LLM call in the agent loop.
#
# Function signature:
#   request: ModelRequest    → everything about this LLM call
#     request.messages       → shortcut for request.state["messages"]
#     request.state          → full graph state dict
#     request.runtime        → context, config, other runtime data
#   handler: Callable        → the actual LLM call to make
#   returns: ModelResponse   → the LLM's response
#
# request.override(model=...) creates a new request object with
# the model field replaced. The original request is immutable —
# override() returns a modified copy. This is intentional: it
# prevents accidental mutation of shared request state.
#
# handler(request) executes the actual LLM API call with the
# (possibly modified) request. Always call this at the end —
# if you forget, the LLM call never happens.
#
# Threshold of 10 messages is the split point:
#   ≤ 10 messages → standard_model (fast, cheap)
#   > 10 messages → large_model (powerful, bigger context window)

from langchain.agents.middleware import wrap_model_call, ModelRequest, ModelResponse
from langchain.chat_models import init_chat_model
from typing import Callable
from langchain_google_genai import ChatGoogleGenerativeAI

# Pre-initialise both models (no API call at this point)
large_model = init_chat_model("claude-sonnet-4-5")  # Powerful: large context window, strong reasoning
standard_model = init_chat_model("gpt-5-nano")       # Efficient: fast and cheap for routine turns
gemini_model = ChatGoogleGenerativeAI(model="gemini-2.5-flash-lite")

@wrap_model_call
def state_based_model(
    request: ModelRequest, 
    handler: Callable[[ModelRequest], ModelResponse]
) -> ModelResponse:
    """Select model based on State conversation length."""

    # request.messages is a convenient shortcut for request.state["messages"]
    intent = request.runtime.context.intent
    if intent == 'Reporting':
        print("Intent is Reporting")
        # Do what ever you want to do
    message_count = len(request.messages)

    if message_count > 10:
        model = large_model    # Long conversation needs more reasoning power
    else:
        model = standard_model # Short conversation — standard model is sufficient

    # Override the model in the request (immutable — creates a new request object)
    request = request.override(model=model)

    # Execute the LLM call with the (possibly modified) model
    return handler(request)

In [ ]:
# @wrap_model_call
# def state_based_model(
#     request: ModelRequest,
#     handler: Callable[[ModelRequest], ModelResponse]
# ) -> ModelResponse:
#     """Select model based on State conversation length."""
#     """
#     model = logic to decide model
#     """
#      # Override the model in the request (immutable — creates a new request object)
#     request = request.override(model=model)
#
#     # Execute the LLM call with the (possibly modified) model
#     return handler(request)


In [4]:
# ============================================================
# CELL 3: Create the Agent With Dynamic Model Middleware
# ============================================================
# The model="gpt-5-nano" in create_agent is the DEFAULT model.
# The middleware intercepts every LLM call and may override it.
# For short conversations (≤10 messages), gpt-5-nano is used as-is.
# For long conversations (>10 messages), the middleware switches to claude.

from langchain.agents import create_agent

@dataclass
class MessageIntent:
    intent: str

agent = create_agent(
    model="gpt-5-nano",                  # Default — may be overridden by middleware
    middleware=[state_based_model],      # Dynamic routing logic
    system_prompt="You are roleplaying a real life helpful office intern."
)

In [5]:
# ============================================================
# CELL 4: Short Conversation — Expect Standard Model
# ============================================================
# One message = 1 message in history.
# 1 ≤ 10 → middleware selects standard_model (gpt-5-nano).

from langchain.messages import HumanMessage

response = agent.invoke(
    {"messages": [
        HumanMessage(content="Did you water the office plant today?")
    ]},
    context = MessageIntent('Reporting')
)

print(response["messages"][-1].content)

Intent is Reporting
Yes—I watered the office plant this morning. Want me to set a reminder for the next watering or add it to a little plant care log?


In [ ]:
# ============================================================
# CELL 5: Verify Which Model Was Used
# ============================================================
# response_metadata is returned by every LLM call and includes:
#   'model_name' → the exact model that processed this request
#
# For a short conversation, this should show 'gpt-5-nano-...'.
# This is how you verify the dynamic routing worked correctly.

print(response["messages"][-1].response_metadata["model_name"])

In [ ]:
# ============================================================
# CELL 6: Long Conversation — Expect Large Model
# ============================================================
# We inject 11 messages (10 existing + 1 new question).
# 11 > 10 → middleware selects large_model (claude-sonnet-4-5).
#
# Notice we inject prior Human/AI pairs to simulate a long conversation.
# In a real app with checkpointer, this history would accumulate naturally.
# Here we inject it to trigger the threshold in a single test call.

from langchain.messages import AIMessage

response = agent.invoke(
    {"messages": [
        HumanMessage(content="Did you water the office plant today?"),
        AIMessage(content="Yes, I gave it a light watering this morning."),
        HumanMessage(content="Has it grown much this week?"),
        AIMessage(content="It's sprouted two new leaves since Monday."),
        HumanMessage(content="Are the leaves still turning yellow on the edges?"),
        AIMessage(content="A little, but it's looking healthier overall."),
        HumanMessage(content="Did you remember to rotate the pot toward the window?"),
        AIMessage(content="I rotated it a quarter turn so it gets more even light."),
        HumanMessage(content="How often should we be fertilizing this plant?"),
        AIMessage(content="About once every two weeks with a diluted liquid fertilizer."),
        HumanMessage(content="When should we expect to have to replace the pot?")  # 11th message
    ]}
)

print(response["messages"][-1].content)

In [ ]:
# ============================================================
# CELL 7: Verify Model Switched to Claude
# ============================================================
# For the long conversation (11 messages > threshold of 10),
# this should show a Claude model name (claude-sonnet-4-5-...).
# Compare with Cell 5 — same agent, different model selected
# purely based on conversation length.

print(response["messages"][-1].response_metadata["model_name"])

---
## Summary

| Concept | Key takeaway |
|---|---|
| **`@wrap_model_call`** | Intercepts every LLM call — can inspect, modify, and re-route |
| **`request.messages`** | Shortcut for `request.state["messages"]` — the current conversation |
| **`request.override(model=...)`** | Returns a new request with a different model — immutable, safe |
| **`handler(request)`** | Executes the actual LLM call — always call this at the end |
| **`response_metadata["model_name"]`** | Verify which model actually answered |
| **Pre-initialise models** | Create model objects once at module load — don't recreate per call |

### Other things you can override besides model

```python
request.override(model=large_model)          # Switch the LLM
request.override(tools=[tool_a, tool_b])     # Change available tools
request.override(temperature=0.0)            # Force determinism for this call
request.override(max_tokens=500)             # Limit response length
```

### Next steps
- **3.4 Dynamic Prompts** — change the system prompt per request based on context
- **3.4 Dynamic Tools** — restrict tool access per user role